# High-altitude set: where the pipeline fails, stage by stage (2026-11-17)

Replay of `results/highalt_truth_diag/` (no GPU needed). The tables were made by `scripts/highalt_truth_diag/` (see its README): one query per photo
(heading 0, scale 1.0) through the submission pipeline's own stages, with the true place known.

Limits: one satellite tile (2684532111), so the search disc is cut for photos near its south edge; one query per photo, not six; map levels from a 1.12 m copy of the tile.
Splits are the ones in `results/highalt_split.csv`. Thresholds are taken from the dev split and applied to the test split unchanged.

In [1]:
import glob, os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
R = "../results/highalt_truth_diag/"
def points(ratio):
    return np.where(ratio <= 1, 10, np.where(ratio <= 2, 6, np.where(ratio <= 4, 3, np.where(ratio <= 8, 0, -20))))
base = pd.read_csv(R + "diag_base.csv")
base = base[base.truth_z > -50].copy()                    # drop photos whose true cell is outside the tile
base["coarse_first"] = base.truth_rank_cells == 0         # the true place is the best cell of the whole disc
bands = [150, 400, 600, 800, 1000, 1200, 1400, 1600, 2100]
len(base)

447

## 1. Coarse search against LoFTR at the true place, by height

`coarse_first`: share of photos where the true place has the highest edge-correlation score of the whole disc.
`loftr_8` / `loftr_4` / `loftr_0`: share where LoFTR, given the true place, reaches 8 inliers, 4 inliers, or none.

In [2]:
g = base.groupby(pd.cut(base.alt, bands), observed=True)
pd.DataFrame(dict(photos=g.size(), coarse_first=g.coarse_first.mean(), truth_z=g.truth_z.median(), best_wrong_z=g.max_far_z.median(),
                  loftr_inliers=g.inl_truth.median(), loftr_8=g.apply(lambda x: (x.inl_truth >= 8).mean()),
                  loftr_4=g.apply(lambda x: (x.inl_truth >= 4).mean()), loftr_0=g.apply(lambda x: (x.inl_truth == 0).mean()))).round(2)

,photos,coarse_first,truth_z,best_wrong_z,loftr_inliers,loftr_8,loftr_4,loftr_0
alt,,,,,,,,
"(150, 400]",94,0.96,12.95,5.60,47.5,0.97,0.99,0.01
"(400, 600]",49,0.96,11.36,4.95,47.0,0.98,1.00,0.00
"(600, 800]",93,1.00,11.24,4.43,60.0,0.96,0.98,0.02
"(800, 1000]",23,0.91,8.06,4.37,33.0,0.78,0.91,0.09
"(1000, 1200]",38,0.84,8.31,4.25,20.0,0.71,0.74,0.24
"(1200, 1400]",48,0.96,8.41,4.21,10.5,0.58,0.77,0.23
"(1400, 1600]",40,0.88,7.14,4.25,0.0,0.38,0.42,0.55
"(1600, 2100]",62,0.60,4.80,4.10,0.0,0.03,0.13,0.84


## 2. Photo preparation and canvas variants (dev photos above 800 m, 142 photos)

`flat` = radial vignette fit removed + corner artefact masked; `flatnorm` = flat + local normalisation (sigma 15 px); `noclahe` = stretch without CLAHE;
`lm1` / `lp1` = one map level finer / coarser on the 224 canvas; `t336`, `t448` = larger canvas (level follows); `l12` = level 12 allowed.

`coarse_first` as above. `loftr_8` = LoFTR reaches 8 inliers at the true place. `pipeline_8` = the query's own best candidate has 8 inliers and is right.
`coarse_rule` = share of photos answered when the coarse peak must beat the highest wrong coarse peak of this table (in-sample, so an upper bound).

In [3]:
rows = []
for path in sorted(glob.glob(R + "var_*.csv")):
    if "_TEST" in path: continue
    d = pd.read_csv(path); wrong = d[~d.top1_right]
    rows.append(dict(variant=os.path.basename(path)[4:-4], coarse_first=d.top1_right.mean(), coarse_first_1600=d[d.alt >= 1600].top1_right.mean(),
                     loftr_8=(d.inl_truth >= 8).mean(), loftr_8_1600=(d[d.alt >= 1600].inl_truth >= 8).mean(),
                     pipeline_8=((d.best_inl >= 8) & (d.best_err_sig <= 8)).mean(), wrong_inliers_max=d.max_inl_wrong.max(),
                     threshold_z=wrong.p1z.max(), coarse_rule=(d.top1_right & (d.p1z > wrong.p1z.max())).mean()))
pd.DataFrame(rows).round(2)

,variant,coarse_first,coarse_first_1600,loftr_8,loftr_8_1600,pipeline_8,wrong_inliers_max,threshold_z,coarse_rule
0,base,0.75,0.50,0.37,0.00,0.38,0.0,5.61,0.64
1,flat,0.80,0.52,0.35,0.05,0.36,4.0,5.70,0.65
2,flatnorm,0.83,0.64,0.35,0.02,0.35,0.0,5.42,0.75
3,l12,0.75,0.50,0.37,0.00,0.37,0.0,5.61,0.62
4,l12flatnorm,0.83,0.64,0.34,0.00,0.35,0.0,5.42,0.76
5,lm1,0.80,0.45,0.41,0.02,0.46,0.0,7.29,0.63
6,lm1flatnorm,0.79,0.45,0.42,0.02,0.40,0.0,8.02,0.57
7,lp1,0.72,0.50,0.33,0.00,0.35,3.0,5.03,0.63
8,noclahe,0.70,0.48,0.30,0.00,0.32,0.0,5.48,0.58
9,t336,0.75,0.43,0.35,0.07,0.44,0.0,9.41,0.39


## 3. Coarse-evidence rule on the held-out test split (72 photos above 800 m)

Rule: answer at the best coarse peak when its z-score over the disc is above the threshold found on the dev split (base 5.61, flatnorm 5.42), position refined by a pixel-step correlation.
`LoFTR` = the one query's best candidate with at least 8 inliers. `both` = LoFTR's answer when it has one, otherwise the coarse answer.

In [4]:
rows = []; by_band = {}
for tag, thr in (("base", 5.61), ("flatnorm", 5.42)):
    d = pd.read_csv(R + "var_%s_TEST.csv" % tag)
    lo = d.best_inl >= 8; lp = np.where(lo, points(d.best_err_sig), 0)
    for margin in (0.0, 0.5):
        co = d.p1z > thr + margin; cp = np.where(co, points(d.ref_err / d.sigma), 0); both = np.where(lo, lp, cp)
        rows.append(dict(photo=tag, threshold=thr + margin, photos=len(d), loftr_answered=lo.sum(), loftr_points=lp.mean(), coarse_answered=co.sum(),
                         coarse_wrong_place=(co & ~d.top1_right).sum(), coarse_points=cp.mean(), both_answered=(lo | co).sum(), both_points=both.mean(), both_below_zero=(both < 0).sum()))
    co = d.p1z > thr
    t = pd.DataFrame(dict(band=pd.cut(d.alt, [800, 1200, 1600, 2100]), lo=lo, co=co, lp=lp, cp=np.where(co, points(d.ref_err / d.sigma), 0)))
    t["both"] = np.where(t.lo, t.lp, t.cp)
    by_band[tag] = t.groupby("band", observed=True).agg(photos=("lo", "size"), loftr_answered=("lo", "mean"), coarse_answered=("co", "mean"),
                                                         loftr_points=("lp", "mean"), coarse_points=("cp", "mean"), both_points=("both", "mean")).round(2)
pd.DataFrame(rows).round(2)

,photo,threshold,photos,loftr_answered,loftr_points,coarse_answered,coarse_wrong_place,coarse_points,both_answered,both_points,both_below_zero
0,base,5.61,72,38,3.33,50,0,4.24,55,4.83,0
1,base,6.11,72,38,3.33,48,0,4.11,54,4.75,0
2,flatnorm,5.42,72,35,2.93,61,0,5.19,63,5.44,0
3,flatnorm,5.92,72,35,2.93,57,0,4.94,61,5.32,0


In [5]:
print("base photo"); print(by_band["base"]); print(); print("flatnorm photo"); print(by_band["flatnorm"])
d = pd.read_csv(R + "var_flatnorm_TEST.csv"); a = d[(d.p1z > 5.42) & d.top1_right]; r = a.ref_err / a.sigma
print("\nposition error of the coarse answers (flatnorm), in sigma: median %.2f; within 1 / 2 / 4 sigma: %.2f / %.2f / %.2f" % (r.median(), (r <= 1).mean(), (r <= 2).mean(), (r <= 4).mean()))

base photo
              photos  loftr_answered  coarse_answered  loftr_points  coarse_points  both_points
band                                                                                           
(800, 1200]       25            0.80             0.76          4.96           3.96         5.08
(1200, 1600]      27            0.59             0.78          3.96           5.48         5.81
(1600, 2100]      20            0.10             0.50          0.45           2.90         3.20

flatnorm photo
              photos  loftr_answered  coarse_answered  loftr_points  coarse_points  both_points
band                                                                                           
(800, 1200]       25            0.76             0.96          4.00           5.32         5.52
(1200, 1600]      27            0.48             0.93          3.56           6.67         6.93
(1600, 2100]      20            0.15             0.60          0.75           3.05         3.35

position err

## 4. The same rule below 800 m, and with the number of cells taken into account

Below 800 m the disc has many more cells, so a wrong peak reaches a higher z by chance. `ev` = z minus sqrt(2 ln(cells)).

In [6]:
base["p1z"] = np.where(base.coarse_first, base.truth_z, base.max_far_z); base["ev"] = base.p1z - np.sqrt(2 * np.log(base.n_cells)); rows = []
for col in ("p1z", "ev"):
    for lo, hi in ((150, 800), (800, 2100)):
        dev = base[(base.split == "dev") & (base.alt >= lo) & (base.alt < hi)]; test = base[(base.split == "test") & (base.alt >= lo) & (base.alt < hi)]
        thr = dev[~dev.coarse_first][col].max()
        for name, t, th in (("dev", dev, thr), ("test", test, thr), ("test, +0.5", test, thr + 0.5)):
            a = t[t[col] > th]
            rows.append(dict(score=col, heights="%d-%d" % (lo, hi), split=name, threshold=th, photos=len(t), answered=len(a), wrong_place=(~a.coarse_first).sum(), right_with_loftr_under_8=(a.coarse_first & (a.inl_truth < 8)).sum()))
pd.DataFrame(rows).round(2)

,score,heights,split,threshold,photos,answered,wrong_place,right_with_loftr_under_8
0,p1z,150-800,dev,5.75,153,152,0,2
1,p1z,150-800,test,5.75,83,79,3,4
2,p1z,150-800,"test, +0.5",6.25,83,74,0,4
3,p1z,800-2100,dev,5.61,139,89,0,41
4,p1z,800-2100,test,5.61,72,48,0,16
5,p1z,800-2100,"test, +0.5",6.11,72,48,0,16
6,ev,150-800,dev,0.49,153,152,0,2
7,ev,150-800,test,0.49,83,79,3,4
8,ev,150-800,"test, +0.5",0.99,83,75,1,4
9,ev,800-2100,dev,0.76,139,91,0,43


## 5. Position error of LoFTR answers at the true place: a pointing offset of this camera

For photos with at least 8 inliers at the true place, the error is split into forward / right of the camera (heading = yaw + 180) and turned into an angle (error / height).
A constant offset, and a fit linear in pitch and roll, are taken from the dev split and applied to the test split.

In [7]:
d = base[base.inl_truth >= 8].copy(); hd = np.radians(d.yaw + 180.0); E = d.sh_e.values; N = -d.sh_s.values
d["fwd"] = E * np.sin(hd) + N * np.cos(hd); d["rgt"] = E * np.cos(hd) - N * np.sin(hd); H = d.alt * 1.04
d["fwd_deg"] = np.degrees(np.arctan(d.fwd / H)); d["rgt_deg"] = np.degrees(np.arctan(d.rgt / H))
print(d.groupby(pd.cut(d.alt, [150, 400, 800, 1200, 1600]), observed=True)[["fwd_deg", "rgt_deg"]].agg(["median", "count"]).round(2)); print()
print(d.groupby(pd.cut(d.pitch, [-10, -3, 0, 2, 3.5, 4.5, 10]), observed=True)[["fwd_deg", "rgt_deg"]].agg(["median", "count"]).round(2)); print()
print(d.groupby(pd.cut(d.roll, [-25, -10, -5, -1, 1, 5, 10, 25]), observed=True)[["fwd_deg", "rgt_deg"]].agg(["median", "count"]).round(2))
dev, test = d[d.split == "dev"], d[d.split == "test"]; rows = []
def design(t): return np.c_[np.ones(len(t)), np.radians(t.pitch), np.radians(t.roll)]
for kind in ("constant", "linear in pitch and roll"):
    if kind == "constant":
        cf = np.array([np.median(dev.fwd / (dev.alt * 1.04)), 0, 0]); cr = np.array([np.median(dev.rgt / (dev.alt * 1.04)), 0, 0])
    else:
        w = dev.inl_truth >= 15
        cf = np.linalg.lstsq(design(dev[w]), (dev.fwd / (dev.alt * 1.04))[w], rcond=None)[0]; cr = np.linalg.lstsq(design(dev[w]), (dev.rgt / (dev.alt * 1.04))[w], rcond=None)[0]
    for name, t in (("dev", dev), ("test", test)):
        Ht = t.alt * 1.04; before = np.hypot(t.fwd, t.rgt) / t.sigma; after = np.hypot(t.fwd - Ht * (design(t) @ cf), t.rgt - Ht * (design(t) @ cr)) / t.sigma
        for lo, hi in ((150, 800), (800, 1200), (1200, 2100)):
            m = (t.alt >= lo) & (t.alt < hi)
            rows.append(dict(model=kind, split=name, heights="%d-%d" % (lo, hi), photos=m.sum(), points_before=points(before[m]).mean(), points_after=points(after[m]).mean(), error_sigma_before=before[m].median(), error_sigma_after=after[m].median()))
pd.DataFrame(rows).round(2)

             fwd_deg       rgt_deg      
              median count  median count
alt                                     
(150, 400]     -0.53    91   -0.49    91
(400, 800]     -0.55   137   -0.10   137
(800, 1200]    -0.61    45   -0.43    45
(1200, 1600]   -0.03    43   -0.37    43

              fwd_deg       rgt_deg      
               median count  median count
pitch                                    
(-10.0, -3.0]   -1.76    57   -0.45    57
(-3.0, 0.0]      0.06    19   -0.76    19
(0.0, 2.0]      -0.23    20   -0.33    20
(2.0, 3.5]      -0.46   125   -0.08   125
(3.5, 4.5]      -0.31    88   -0.50    88
(4.5, 10.0]     -0.61     9    0.23     9

           fwd_deg       rgt_deg      
            median count  median count
roll                                  
(-25, -10]   -1.24    58    0.33    58
(-10, -5]    -0.50    59    0.23    59
(-5, -1]     -0.39    98   -0.39    98
(-1, 1]      -0.04    40   -0.60    40
(1, 5]       -0.37    17   -0.76    17
(5, 10]      -1.49   

,model,split,heights,photos,points_before,points_after,error_sigma_before,error_sigma_after
0,constant,dev,150-800,150,8.53,8.96,0.68,0.58
1,constant,dev,800-1200,23,7.22,8.65,1.03,0.84
2,constant,dev,1200-2100,29,8.66,8.66,0.74,0.88
3,constant,test,150-800,78,8.35,8.79,0.56,0.48
4,constant,test,800-1200,22,6.55,7.23,1.37,1.07
5,constant,test,1200-2100,16,8.56,7.31,0.90,1.03
6,linear in pitch and roll,dev,150-800,150,8.53,9.35,0.68,0.48
7,linear in pitch and roll,dev,800-1200,23,7.22,8.96,1.03,0.64
8,linear in pitch and roll,dev,1200-2100,29,8.66,8.10,0.74,0.92
9,linear in pitch and roll,test,150-800,78,8.35,9.24,0.56,0.49


## What the tables say

- Above 800 m the coarse search still puts the true place first for most photos while LoFTR, even when handed the true place, does not reach the bar; above 1600 m LoFTR gets nothing in any variant.
- Removing the vignette does not help LoFTR (CLAHE already flattens it); with local normalisation it helps the coarse search. A larger canvas makes things worse; one level finer helps a little at 1200-1600 m.
- A rule on the coarse peak alone, with the threshold from the dev split, answers more of the held-out photos above 800 m than LoFTR with no wrong place. Below 800 m the same rule gives wrong places, so it is for the high levels only.
- Coarse answers are mostly 1 to 2 sigma off, so they earn 6 more often than 10.
- LoFTR answers carry a pointing offset of this camera; a constant or linear correction from the dev split helps the test split below 1200 m and not above.

Not shown here and still to do: six queries instead of one, the three-tile store, a second region.